# Semantic extraction with Azure Content Understanding

Create a synthetic purchase-order PDF, define a custom semantic analyzer, and extract named fields rather than only OCR text. Display the returned values, confidence, encoded source grounding, and supporting markdown spans. Save submission handles before polling so a later Spark session can resume without uploading the document again.

This notebook uses SynapseML, PySpark, and the Python standard library. It does **not** install or call the CU Python SDK. Requests are billable and analyzer creation is an explicit resource modification; use a development CU resource. No real customer document is needed.

## 1. Configure your runtime and resource

Start a Spark session with matching SynapseML Python and JVM artifacts from the same build containing `ContentUnderstanding`. A preinstalled older SynapseML version may not contain the connector. Follow the [SynapseML installation instructions](../../Get%20Started/Install%20SynapseML.md) for your runtime before running these cells; adding only the Python package is insufficient. No additional document-generation or CU SDK package is required.

Use an existing CU resource with access to `prebuilt-document`, its required model deployments/defaults, and permission to create an analyzer. This notebook does not provision a resource or change its model defaults. See the [REST quickstart](https://learn.microsoft.com/azure/ai-services/content-understanding/quickstart/use-rest-api) for resource/model setup.

Set the following environment variables in the notebook's driver process, or answer the prompts. Inject the key from your platform's secret manager; do not put it in a notebook cell or output.

| Setting | Value |
| --- | --- |
| `CONTENT_UNDERSTANDING_ENDPOINT` | Your CU resource root, such as `https://<resource>.services.ai.azure.com`. |
| `CONTENT_UNDERSTANDING_API_KEY` | Resource key, obtained from a secret manager; otherwise prompted without echo. |
| `CONTENT_UNDERSTANDING_OUTPUT_PATH` | Durable Spark-readable/writable directory accessible to executors. For an attached Fabric Lakehouse, for example, `Files/cu-semantic-example`; otherwise use your cluster's configured storage URI. Do not use driver-local storage on a distributed cluster. |
| `CONTENT_UNDERSTANDING_RUN_ID` | Optional stable run identifier: 1-39 letters, digits, hyphens or underscores, starting with a letter or digit. Omit for a new run; reuse the printed identifier for recovery. |
| `CONTENT_UNDERSTANDING_MODEL_DEPLOYMENTS` | Optional JSON object mapping model names to deployment names, sent as `modelDeployments` with analysis submissions. It does not change analyzer creation or resource defaults. Blank uses resource defaults; recovery does not read this setting. It is validated before analyzer creation to fail before service calls. |

All input data and derived storage paths are created below. The API version is explicitly GA `2025-11-01`.

In [ ]:
import getpass
import json
import os
import re
import uuid
from urllib.parse import urlsplit

from pyspark.sql import SparkSession, functions as F
from synapse.ml.services.contentunderstanding import ContentUnderstanding

spark = SparkSession.builder.getOrCreate()
endpoint = (
    os.environ.get("CONTENT_UNDERSTANDING_ENDPOINT")
    or input("CU resource endpoint: ").strip()
)
key = os.environ.get("CONTENT_UNDERSTANDING_API_KEY") or getpass.getpass(
    "CU resource key (not displayed): "
)
storage_root = (
    os.environ.get("CONTENT_UNDERSTANDING_OUTPUT_PATH")
    or input("Durable Spark output directory: ").strip()
)
run_id = (
    os.environ.get("CONTENT_UNDERSTANDING_RUN_ID", "").strip() or uuid.uuid4().hex[:12]
)
if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_-]{0,38}", run_id):
    raise ValueError(
        "Run ID must be 1-39 letters, digits, hyphens or underscores, "
        "starting with a letter or digit."
    )
if not endpoint or not key or not storage_root:
    raise ValueError("An endpoint, resource key, and output directory are required.")


def resource_origin(url: str) -> str:
    try:
        parsed = urlsplit(url)
        if (
            parsed.scheme not in ("http", "https")
            or not parsed.hostname
            or parsed.username is not None
            or parsed.password is not None
        ):
            return ""
        port = parsed.port
        if port is None:
            port = 443 if parsed.scheme == "https" else 80
        return f"{parsed.scheme}://{parsed.hostname}:{port}"
    except ValueError:
        return ""


endpoint_origin = resource_origin(endpoint)
if not endpoint_origin:
    raise ValueError("Configure a valid CU resource endpoint without user information.")

run_root = storage_root.rstrip("/") + "/" + run_id
handles_path = run_root + "/submitted"
analyzer_id = "synapseml-purchase-order-" + run_id
api_version = "2025-11-01"
max_poll_attempts = 120
max_operation_duration_ms = 600000
polling_delay_ms = 1000

print("Run identifier:", run_id)
print("Saved submission path:", handles_path)
print("Custom analyzer identifier:", analyzer_id)

## 2. Create a sample document

The following small, single-page PDF is generated locally without a PDF library or external download. Its fictional order contains a supplier, amount, and requested delivery date. The custom analyzer will also generate a short order summary, demonstrating semantic interpretation beyond `prebuilt-read` OCR.

In [ ]:
page_text = b"""BT /F1 14 Tf 50 740 Td
(SYNTHETIC PURCHASE ORDER PO-1001) Tj
0 -30 Td (Supplier: Contoso Office Supplies) Tj
0 -30 Td (Buyer: Fabrikam Research) Tj
0 -30 Td (Order date: 2026-01-15) Tj
0 -30 Td (Items: 10 notebooks at USD 12.50 each.) Tj
0 -30 Td (Total amount: USD 125.00) Tj
0 -30 Td (Delivery requested by 2026-02-01.) Tj
ET"""
pdf_objects = [
    b"<< /Type /Catalog /Pages 2 0 R >>",
    b"<< /Type /Pages /Kids [4 0 R] /Count 1 >>",
    b"<< /Type /Font /Subtype /Type1 /BaseFont /Helvetica >>",
    b"<< /Type /Page /Parent 2 0 R /MediaBox [0 0 612 792] "
    b"/Resources << /Font << /F1 3 0 R >> >> /Contents 5 0 R >>",
    f"<< /Length {len(page_text)} >>\nstream\n".encode("ascii")
    + page_text
    + b"\nendstream",
]
pdf_bytes = bytearray(b"%PDF-1.4\n")
offsets = [0]
for number, pdf_object in enumerate(pdf_objects, 1):
    offsets.append(len(pdf_bytes))
    pdf_bytes.extend(f"{number} 0 obj\n".encode("ascii") + pdf_object + b"\nendobj\n")
xref_offset = len(pdf_bytes)
pdf_bytes.extend(f"xref\n0 {len(offsets)}\n0000000000 65535 f \n".encode("ascii"))
for offset in offsets[1:]:
    pdf_bytes.extend(f"{offset:010d} 00000 n \n".encode("ascii"))
pdf_bytes.extend(
    (
        f"trailer\n<< /Size {len(offsets)} /Root 1 0 R >>\n"
        f"startxref\n{xref_offset}\n%%EOF\n"
    ).encode("ascii")
)

documents = spark.createDataFrame(
    [("PO-1001", "purchase-order.pdf", pdf_bytes)],
    "documentId string, documentName string, content binary",
).coalesce(1)

## 3. Create an explicit semantic analyzer

The field schema describes the desired meaning and types. `estimateFieldSourceAndConfidence` requests confidence and grounding; `returnDetails` retains document details. Generated summaries may not have a confidence or direct source span, so missing values must remain missing rather than being invented.

Run this creation cell once for a **new** run. `allowReplace=False` prevents overwriting an existing analyzer. For recovery of saved analysis handles, skip sample generation, analyzer creation, and submission; polling requires only the endpoint, credentials, and saved results. If creation itself fails or times out, inspect the analyzer through `getAnalyzer()` before considering another creation attempt; analysis poll mode cannot resume an analyzer-management handle.

In [ ]:
try:
    model_deployments = json.loads(
        os.environ.get("CONTENT_UNDERSTANDING_MODEL_DEPLOYMENTS", "").strip() or "{}"
    )
except json.JSONDecodeError as error:
    raise ValueError(
        "Model deployments must be a JSON object of nonempty strings."
    ) from error
if not isinstance(model_deployments, dict) or any(
    not isinstance(name, str)
    or not name.strip()
    or not isinstance(deployment, str)
    or not deployment.strip()
    for name, deployment in model_deployments.items()
):
    raise ValueError("Model deployments must be a JSON object of nonempty strings.")

definition = {
    "baseAnalyzerId": "prebuilt-document",
    "description": "Extract a synthetic purchase order and summarize its purpose.",
    "config": {
        "returnDetails": True,
        "estimateFieldSourceAndConfidence": True,
    },
    "fieldSchema": {
        "name": "PurchaseOrder",
        "fields": {
            "Supplier": {
                "type": "string",
                "description": "The company supplying the ordered goods, not the buyer.",
                "method": "extract",
            },
            "TotalAmount": {
                "type": "number",
                "description": "The total purchase-order amount, excluding the currency label.",
                "method": "extract",
            },
            "RequestedDeliveryDate": {
                "type": "date",
                "description": "The requested delivery date, not the order creation date.",
                "method": "extract",
            },
            "OrderSummary": {
                "type": "string",
                "description": "Summarize in one sentence who is buying what, from whom, and by when.",
                "method": "generate",
            },
        },
    },
}
analyzer = (
    ContentUnderstanding()
    .setEndpoint(endpoint)
    .setSubscriptionKey(key)
    .setApiVersion(api_version)
    .setAnalyzerId(analyzer_id)
    .setDocumentBytesCol("content")
    .setDocumentNameCol("documentName")
    .setMimeType("application/pdf")
    .setStringEncoding("codePoint")
    .setOutputCol("analysis")
    .setErrorCol("requestError")
    .setMaxPollAttempts(max_poll_attempts)
    .setMaxOperationDuration(max_operation_duration_ms)
    .setPollingDelay(polling_delay_ms)
)
if model_deployments:
    analyzer.setModelDeployments(model_deployments)
created = json.loads(analyzer.createAnalyzer(definition, allowReplace=False))
if created.get("status", "").lower() != "ready":
    raise RuntimeError(
        "The analyzer is not ready; inspect it before submitting documents."
    )
print("Analyzer ready:", created["analyzerId"])

## 4. Submit once and save the handles

The write is the Spark action that sends the document. Do not display or collect the live transformed DataFrame before or after this write. Read the stored result instead. This snapshot keeps row identity, the response, and diagnostics without requiring the original PDF for recovery.

`errorifexists` prevents overwriting a previous snapshot at this path, but it does not provide exactly-once submission: task retries or failure between service acceptance and storage can still duplicate work. Do not blindly rerun submission after an ambiguous failure.

In [ ]:
submitter = analyzer.copy({}).setOperationMode("submit")
submitter.transform(documents).select(
    "documentId", "analysis", "requestError"
).write.mode("errorifexists").parquet(handles_path)

submitted = spark.read.parquet(handles_path)
submitted.select("documentId", "analysis.status", "analysis.httpStatus").show(
    truncate=False
)

## 5. Poll saved handles, including after a session restart

On the first pass, read `submitted` and write a separate `poll-001` snapshot. If results remain `NotStarted`, `Running`, or `Unknown` with a non-null handle, set the input below to the last snapshot and choose a new output such as `poll-002`; rerun this section and the display section, **not submission**. Completed, failed, and other non-resumable rows are retained in each snapshot.

After a session restart, run configuration with `CONTENT_UNDERSTANDING_RUN_ID` set to the original printed identifier, the same storage root, and the **same CU resource endpoint**, then jump directly here. The preflight below rejects mismatched handle origins before polling or writing a new snapshot, leaving the saved handles intact. No analyzer creation, model deployment configuration, or original document is needed. The handle supplies the original API version. Do not automatically resubmit `Unknown` rows without a handle; their acceptance is uncertain. `ResultUnavailable` means the saved handle can no longer retrieve the result and needs manual investigation.

Polling happens inside Spark tasks during the write below, not in a detached background service. Each pass has the configured attempt and duration limits. Cancellation stops local work, not the CU operation; resume from the last successfully saved snapshot. By default, a POST 429 returns a rejection without automatically resubmitting, and polling retries count against the limits. To opt in to unlimited cancellable 429 recovery, set `setRetryThrottledRequests(True)` on the analyzer and poller; definitive throttled attempts and their waits then do not consume those budgets. Uncertain submissions are never replayed.

In [ ]:
recovery_input_path = handles_path
recovery_output_path = run_root + "/poll-001"

In [ ]:
saved = spark.read.parquet(recovery_input_path)
resumable = F.col("analysis.operationLocation").isNotNull() & F.col(
    "analysis.status"
).isin("NotStarted", "Running", "Unknown")
pending = saved.where(resumable).select(
    "documentId", F.col("analysis.operationLocation").alias("operationHandle")
)
handle_origin = F.udf(resource_origin, "string")
if (
    pending.where(~handle_origin("operationHandle").eqNullSafe(F.lit(endpoint_origin)))
    .limit(1)
    .count()
):
    raise ValueError(
        "Saved handles do not match the configured CU endpoint. "
        "Restore the original endpoint; no new snapshot was written."
    )
settled = saved.where(~resumable).select("documentId", "analysis", "requestError")
poller = (
    ContentUnderstanding()
    .setEndpoint(endpoint)
    .setSubscriptionKey(key)
    .setOperationMode("poll")
    .setOperationLocationCol("operationHandle")
    .setOutputCol("analysis")
    .setErrorCol("requestError")
    .setMaxPollAttempts(max_poll_attempts)
    .setMaxOperationDuration(max_operation_duration_ms)
    .setPollingDelay(polling_delay_ms)
)
polled = poller.transform(pending).select("documentId", "analysis", "requestError")
settled.unionByName(polled).write.mode("errorifexists").parquet(recovery_output_path)

results = spark.read.parquet(recovery_output_path)
results.selectExpr(
    "documentId",
    "analysis.status AS status",
    "analysis.httpStatus AS httpStatus",
    "get_json_object(analysis.error, '$.code') AS diagnosticCode",
).show(truncate=False)

## 6. Display semantic fields, confidence, and grounding from saved results

Only `Succeeded` establishes success. The following projection covers this example's string, number, and date fields across **all** returned content entries; `rawResponse` in Parquet remains the authoritative, complete result. Extend the schema for other field types rather than assuming every analyzer has this shape.

`source` is CU's encoded grounding (for example, document page/bounding-polygon information); `spans` identify supporting ranges in that content's markdown. `supportingText` slices those ranges using the requested `codePoint` encoding. Confidence is service-estimated, not an accuracy guarantee. Missing values, confidence, or grounding remain null, including when a generated summary has no direct source.

For the synthetic order, inspect whether the supplier is Contoso Office Supplies, the amount is numerically 125 (displayed as `125.0` by the projection), and the requested delivery date is 2026-02-01. The table below is produced from actual returned fields, not those expected values. An empty table is not proof of success: inspect the saved status/diagnostic table above and resume pending handles as needed.

In [ ]:
response_schema = """
STRUCT<result: STRUCT<contents: ARRAY<STRUCT<
    markdown: STRING,
    fields: MAP<STRING, STRUCT<
        type: STRING,
        valueString: STRING,
        valueNumber: DOUBLE,
        valueDate: STRING,
        confidence: DOUBLE,
        source: STRING,
        spans: ARRAY<STRUCT<offset: INT, length: INT>>
    >>
>>>>
"""
completed = results.where("analysis.status = 'Succeeded'")
contents = completed.select(
    "documentId",
    F.posexplode(
        F.from_json("analysis.rawResponse", response_schema)
        .getField("result")
        .getField("contents")
    ).alias("contentIndex", "content"),
)
fields = contents.select(
    "documentId",
    "contentIndex",
    "content.markdown",
    F.explode("content.fields").alias("fieldName", "field"),
)
extracted = fields.select(
    "documentId",
    "contentIndex",
    "fieldName",
    F.col("field.type").alias("type"),
    F.coalesce(
        F.col("field.valueString"),
        F.col("field.valueNumber").cast("string"),
        F.col("field.valueDate"),
    ).alias("value"),
    F.col("field.confidence").alias("confidence"),
    F.col("field.source").alias("source"),
    F.col("field.spans").alias("spans"),
    F.expr(
        "transform(field.spans, span -> substring(markdown, span.offset + 1, span.length))"
    ).alias("supportingText"),
)
extracted.orderBy("documentId", "contentIndex", "fieldName").show(truncate=False)
contents.select("documentId", "contentIndex", "content.markdown").show(truncate=False)

## Retention and references

Keep the last saved snapshot and complete raw results for recovery and audit; service handles can expire. Protect result storage even though this example uses synthetic data. The custom analyzer remains in the resource; remove it with your normal resource-management tooling when no longer needed. This notebook neither deletes analyzers automatically nor changes shared resource defaults.

- [SynapseML Content Understanding guide](Content%20Understanding.md)
- [GA Analyze REST API](https://learn.microsoft.com/en-us/rest/api/contentunderstanding/content-analyzers/analyze?view=rest-contentunderstanding-2025-11-01)
- [Create or replace analyzer REST API](https://learn.microsoft.com/en-us/rest/api/contentunderstanding/content-analyzers/create-or-replace?view=rest-contentunderstanding-2025-11-01)
- [Python SDK reference](https://learn.microsoft.com/en-us/python/api/azure-ai-contentunderstanding/azure.ai.contentunderstanding.contentunderstandingclient?view=azure-python) for comparison only; the notebook does not use it.